In [ ]:
# Load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Path

skyline_path = 'data/MARTHA/DE17501_Martha_results_dotp.csv'
## Read data
skyline_data = pd.read_csv(skyline_path)
# ## For skyline data, remove column Precursor and drop duplicates
# skyline_data = skyline_data.drop(columns=['Precursor'])

# qREPs spike levels
qREPs_spike_levels = pd.read_csv('ratio/DE17501_ratio.csv')

# SDRF 
sdrf_data = pd.read_csv('sdrf/sdrf_MARTHA_combined.sdrf.tsv', sep='\t')
# Save the combined sdrf data to a file
# sdrf_data.to_csv('export/sdrf_MARTHA_combined.tsv', sep='\t', index=False)




In [ ]:
irt_tag_protein = skyline_data[skyline_data['Protein Name'].str.contains('iRT_Tag')] 
iRT_peptides = skyline_data['Peptide Sequence'][skyline_data['Protein Name'].str.contains('iRT_Tag')]
iRT_peptides = iRT_peptides.drop_duplicates().tolist()

print(iRT_peptides)

In [ ]:
# Remove QC samples from skyline_data

qc_samples = sdrf_data[sdrf_data['characteristics[Sample]'].str.contains('QC')]
# File all Replicate that contains QC use Regex to detect QC in Replicate column    
qc_replicates = skyline_data['Replicate'][skyline_data['Replicate'].str.contains('QC')].drop_duplicates().reset_index(drop=True)



# Show all Replicate in df here
print(qc_replicates)
print(f'Number of QC replicates: {len(qc_replicates)}')


# Remove all rows in skyline_data that contains any of the qc_replicates
skyline_data = skyline_data[~skyline_data['Replicate'].isin(qc_replicates)]



In [ ]:
# Plot distrion of Library Dot Product

# Plot distribution of Library Dot Product
plt.figure(figsize=(10, 6))
sns.histplot(skyline_data['Library Dot Product'], bins=20, kde=True)
plt.title('Distribution of Library Dot Product')
plt.xlabel('Library Dot Product')

In [ ]:
import re

# Add column Isotope Label Type
skyline_data['Isotope Label Type'] = skyline_data['Precursor'].apply(lambda x: 'heavy' if re.search(r'heavy', str(x), re.IGNORECASE) else 'light')

# Filter out library dot product that is less than 0.6
skyline_data = skyline_data[skyline_data['Library Dot Product'] > 0.6]

# # Remove Normalied Area that is Nan
skyline_data = skyline_data[skyline_data['Normalized Area'].notna()]
# # Prepare pivot table, keeping all other column names as index except 'Isotope Label Type' and 'Intensity'
index_cols = [col for col in skyline_data.columns if col not in ['Isotope Label Type', 'Intensity']]
skyline_pivot = skyline_data.pivot_table(
    index=['Replicate', 'Peptide'],
    columns='Isotope Label Type',
    values='Normalized Area',
    aggfunc='first'
).reset_index()


In [ ]:
# Summarise each peptide to count how many heavy or light signals are present in skyline_pivot
peptide_counts = skyline_pivot.groupby('Peptide').agg(
    heavy_count=pd.NamedAgg(column='heavy', aggfunc=lambda x: x.notna().sum()),
    light_count=pd.NamedAgg(column='light', aggfunc=lambda x: x.notna().sum())
).reset_index()

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
plt.scatter(peptide_counts['light_count'], peptide_counts['heavy_count'], alpha=0.5)
plt.axvline(700, color='g', linestyle='--', label='Light count = 700')
plt.axhline(700, color='g', linestyle='--', label='Heavy count = 700')
plt.xlabel('Light Count')
plt.ylabel('Heavy Count')
plt.title('Peptide Detection Counts: Light vs Heavy')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Sumamrise Protein and Peptide that has been detected the least
# Count the number of unique peptides and proteins in the data
num_unique_peptides = skyline_data['Peptide'].nunique()

# If Protein Name is not present in peptide_counts (from pivot), map it from skyline_data
if 'Protein Name' in skyline_data.columns:
    num_unique_proteins = skyline_data['Protein Name'].nunique()
else:
    num_unique_proteins = 0

print(f"Number of unique peptides: {num_unique_peptides}")
print(f"Number of unique proteins: {num_unique_proteins}")

# For peptide_counts, Protein Name is missing, so we need to map peptide->protein using skyline_data
peptide_to_protein = skyline_data[['Peptide', 'Protein Name']].drop_duplicates().set_index('Peptide')['Protein Name']

# Add Protein Name to peptide_counts for peptide-based summaries
peptide_counts = peptide_counts.merge(peptide_to_protein, left_on='Peptide', right_index=True, how='left')

# Peptides detected only once in heavy (rare peptides)
least_detected_peptides = peptide_counts[peptide_counts['heavy_count'] == 1]['Peptide'].nunique()
least_detected_proteins = peptide_counts[peptide_counts['heavy_count'] == 1]['Protein Name'].nunique()

# Counting unique peptides with only 1 detection in light/heavy signal
least_detected_light = peptide_counts[peptide_counts['light_count'] == 1].shape[0]
least_detected_heavy = peptide_counts[peptide_counts['heavy_count'] == 1].shape[0]

print(f"Number of peptides detected only once in heavy: {least_detected_peptides}")
print(f"Number of proteins (containing such peptides) detected only once in heavy: {least_detected_proteins}")

print(f"Number of peptides detected only once in light: {least_detected_light}")
print(f"Number of peptides detected only once in heavy: {least_detected_heavy}")




In [ ]:
# Set the cut off at 700 on both light and heavy count and then summarise the peptide list and plot the scatter again
peptide_counts_cutoff = peptide_counts[(peptide_counts['heavy_count'] > 700) & (peptide_counts['light_count'] > 700)]



# Summarise how many peptide and protein are detected in the cutoff
peptide_counts_cutoff['Peptide'].nunique()
print(f"Number of peptides detected in the cutoff: {peptide_counts_cutoff['Peptide'].nunique()}")

# Fitlered peptides list 
selected_peptides = peptide_counts_cutoff['Peptide'].drop_duplicates().tolist()
peptide_counts_cutoff['Protein Name'].nunique()
print(f"Number of proteins detected in the cutoff: {peptide_counts_cutoff['Protein Name'].nunique()}")

In [ ]:
# Pick theese columns from skyline_data
select_col = ['Replicate', 'Peptide Sequence', 'Protein Name', 'RatioLightToHeavy']

# Filtered skyline_data with selected_peptides
skyline_data_filtered = skyline_data[skyline_data['Peptide'].isin(selected_peptides)]

# Se;ect Replciate, Peptide Sequence, Protein Name and RatioLighToHeavy from skyline_data_filtered
skyline_data_filtered = skyline_data_filtered[select_col].drop_duplicates()


# Merge skyline_data_filtered with sdrf_data using 'Replicate' from skyline and 'comment[data file]' from sdrf_data
skyline_merge = pd.merge(skyline_data_filtered, sdrf_data[['source name', 'characteristics[Sample]']], left_on='Replicate', right_on='source name', how='left')

# Create column characteristics[Plate] from characteristics[Sample] (should be whatever comes after 'Plate_' in Replicate)
skyline_merge['characteristics[Plate]'] = skyline_merge['Replicate'].str.extract(r'Plate_(\d+)')


In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 

pool_data = skyline_merge[skyline_merge['characteristics[Sample]'] == 'Pool']


In [ ]:

# Plot log_ratio of each sample in boxplot, colored by plate, but x-axis is Replicate, sorted by plate (though x labels are hidden).
# First, sort dataframe by Plate
df_sorted = pool_data.sort_values('characteristics[Plate]')
sns.boxplot(
    x='Replicate',
    y='RatioLightToHeavy',
    data=df_sorted,
    hue='characteristics[Plate]'
)
plt.title('Boxplot of log(RatioLightToHeavy) by Replicate (colored by Plate)')
# Don't show x labels
plt.xlabel('')
plt.ylabel('log(RatioLightToHeavy)')
plt.yscale('log')
plt.xticks([], [])  # Remove x tick labels and marks
plt.legend(title='Plate', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()



In [ ]:
pivot = pool_data.pivot(
    index='Peptide Sequence',
    columns='Replicate',
    values='RatioLightToHeavy'
)

# log-transform (assumes RatioLightToHeavy > 0)
heatmap_data = np.log(pivot)

# sort replicates (columns, x-axis) by their mean log-ratio
col_order = heatmap_data.mean(axis=0).sort_values().index

# sort peptides (rows, y-axis) by their mean log-ratio
row_order = heatmap_data.mean(axis=1).sort_values().index

# apply both orders
heatmap_data = heatmap_data.loc[row_order, col_order]

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='viridis')
plt.title('Heatmap of log(RatioLightToHeavy) for Pool')
plt.xlabel('')
plt.ylabel('Peptide Sequence')
plt.xticks([], [])  # Remove x tick labels and marks
plt.show()

In [ ]:
# Calculate inter-plate CV per peptide and summarise in a dataframe

# Group by Plate and Peptide Sequence, calculate mean and std for each, then compute intra-plate CV
peptide_plate_stats = (
    pool_data.groupby(['characteristics[Plate]', 'Peptide Sequence'])['RatioLightToHeavy']
    .agg(['mean', 'std'])
    .reset_index()
)
peptide_plate_stats['intra_plate_cv'] = peptide_plate_stats['std'] / peptide_plate_stats['mean']

# Show summary dataframe
display(peptide_plate_stats.head())


plt.figure(figsize=(10, 6))
sns.boxplot(x='characteristics[Plate]', y='intra_plate_cv', data=peptide_plate_stats)
plt.title('Boxplot of Intra Plate CV for Pool')
plt.xlabel('Plate')
plt.ylabel('Intra Plate CV')
plt.show()


In [ ]:
# From pool data, plot box plot of ratio 
# Now, summarise inter-plate CV for each peptide (i.e., how the mean RatioLightToHeavy varies across plates)


# Group by Peptide Sequence and calculate the mean and sd of RatioLightToHeavy, summarise in df
peptide_means = (
    peptide_plate_stats.groupby('Peptide Sequence')['mean']
    .agg(['mean', 'std'])
    # Calculate inter-plate CV from mean of each peptide and std of each peptide from different plates
    .rename(columns={'mean': 'grand_mean', 'std': 'between_plate_sd'})
    .reset_index()
)

# Calculate inter-plate CV per peptide
peptide_means['inter_plate_cv'] = peptide_means['between_plate_sd'] / peptide_means['grand_mean']

# Display summarised dataframe of inter-plate CV per peptide
peptide_means[['Peptide Sequence', 'inter_plate_cv']].head()

# Plot KDE (Kernel Density Estimate) of inter-plate CV by peptide with vertical median line
plt.figure(figsize=(12, 6))
sns.kdeplot(peptide_means['inter_plate_cv'].dropna(), fill=True)
median_cv = peptide_means['inter_plate_cv'].median()
plt.axvline(median_cv, color='red', linestyle='--', label=f'Median = {median_cv:.2f}')
plt.title('KDE Plot of Inter-Plate CV Across Peptides')
plt.xlabel('Inter-Plate CV')
plt.ylabel('Density')
plt.legend()
plt.show()


In [ ]:
# Plot cumulative peptide numbers across all CV values
cv_sorted = peptide_means[['inter_plate_cv', 'Peptide Sequence']].sort_values('inter_plate_cv').reset_index(drop=True)
cv_sorted['cumulative_count'] = range(1, len(cv_sorted) + 1)

# Print out first 10% of CV and peptide count
print(f"First 10% of CV: {cv_sorted['inter_plate_cv'].iloc[0]:.2f}, Peptide count: {cv_sorted['cumulative_count'].iloc[0]}")
# Print out first 20% of CV and peptide count
print(f"First 20% of CV: {cv_sorted['inter_plate_cv'].iloc[19]:.2f}, Peptide count: {cv_sorted['cumulative_count'].iloc[19]}")


plt.figure(figsize=(10, 6))
plt.plot(cv_sorted['inter_plate_cv'], cv_sorted['cumulative_count'], marker='o', linestyle='-')
plt.xlabel('Inter-Plate CV')
plt.ylabel('Cumulative Number of Peptides')
plt.title('Cumulative Peptide Count by Inter-Plate CV')

# Label every 10 percent of CV
for cv_mark in [0.1 * i for i in range(1, 11)]:
    # Find the closest row where the CV >= cv_mark
    mask = cv_sorted['inter_plate_cv'] >= cv_mark
    if mask.any():
        idx = mask.idxmax()
        x = cv_sorted.at[idx, 'inter_plate_cv']
        y = cv_sorted.at[idx, 'cumulative_count']
        plt.axvline(x, color='gray', linestyle='--', linewidth=0.8)
        plt.text(x, y, f"{int(y)} peptides\n{cv_mark:.1f} CV", va='bottom', ha='left', fontsize=9, color='blue')

plt.tight_layout()
plt.show()

In [ ]:
# Plot CV of each peptide in scatter plot, sorted by CV, and hide y labels entirely
peptide_means_sorted = peptide_means.sort_values('inter_plate_cv', ascending=True)
ax = sns.scatterplot(
    x='inter_plate_cv',
    y='Peptide Sequence',
    data=peptide_means_sorted
)
plt.title('Scatter plot of Inter-Plate CV Across Peptides (sorted)')
plt.xlabel('Inter-Plate CV')
ax.set_ylabel('Peptide Sequence')
ax.set_yticklabels([])
ax.set_yticks([])
plt.show()

In [ ]:
# Batch/Plate adjustment, correcting for each peptide within each plate (batch) 
# using only peptides in the top 10% lowest inter-plate CV for normalization

# Find the threshold for the top 10% lowest inter-plate CV peptides
cv_threshold = peptide_means['inter_plate_cv'].quantile(0.1)
# Get list of peptides in the top 10% lowest inter-plate CV
top10cv_peptides = peptide_means[peptide_means['inter_plate_cv'] <= cv_threshold]['Peptide Sequence'].unique()

# Filter pool_data to include only those peptides for normalization calculation
selected_norm_peptides = pool_data[pool_data['Peptide Sequence'].isin(top10cv_peptides)]


In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# 1. Make a clean copy with simple column names
df = selected_norm_peptides.rename(
    columns={
        'characteristics[Plate]': 'Plate',
        'Peptide Sequence': 'Peptide',
    }
).copy()

# 2. Ensure numeric response and drop NAs
df['RatioLightToHeavy'] = pd.to_numeric(df['RatioLightToHeavy'], errors='coerce')
df = df.dropna(subset=['RatioLightToHeavy', 'Plate', 'Peptide'])

# 3. Two-way model: RatioLightToHeavy ~ Plate + Peptide
model = smf.ols('RatioLightToHeavy ~ C(Plate) + C(Peptide)', data=df).fit()
print(model.summary())

# 4. ANOVA to see contributions of Plate vs Peptide
anova_res = anova_lm(model, typ=2)
anova_res

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

df = selected_norm_peptides.rename(
    columns={'characteristics[Plate]': 'Plate'}
).copy()

# Keep only rows we can use
df['RatioLightToHeavy'] = pd.to_numeric(df['RatioLightToHeavy'], errors='coerce')
df = df.dropna(subset=['RatioLightToHeavy', 'Plate'])

# Work on log scale so plate effects are multiplicative factors
df['log_ratio'] = np.log(df['RatioLightToHeavy'])

# Model: log(ratio) ~ Plate (Plate as categorical)
m = smf.ols('log_ratio ~ C(Plate)', data=df).fit()
print(m.summary())
print(f'This summary means that the log(ratio) is different for each plate, and the difference is statistically significant. Even though they are the sample from the pool replicate, there is still a difference in the log(ratio) between the plate.')

In [ ]:
# Plot log_ratio of each sample in boxplot, colored by plate, but x-axis is Replicate, sorted by plate (though x labels are hidden).
# First, sort dataframe by Plate
df_sorted = df.sort_values('Plate')
sns.boxplot(x='Replicate', y='log_ratio', data=df_sorted, hue='Plate')
plt.title('Boxplot of log(RatioLightToHeavy) by Replicate (colored by Plate)')
# Don't show x labels
plt.xlabel('')
plt.ylabel('log(RatioLightToHeavy)')
plt.xticks([], [])  # Remove x tick labels and marks
plt.legend(title='Plate', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Fit a model: log_ratio ~ C(Plate) to extract plate conversion factors
# This will give us the correction (as multiplicative) for each plate

# Refit to get coefficients relative to reference plate
m = smf.ols('log_ratio ~ C(Plate)', data=df).fit()

# Extract coefficients and convert to multiplicative factors
plate_effects = m.params.filter(like='C(Plate)')
# The intercept represents the reference plate's mean log(ratio)
reference_plate = df['Plate'].unique()
if isinstance(df['Plate'].iloc[0], str):
    # some columns come in as string/mixed types
    reference_plate = sorted(reference_plate, key=lambda x: str(x))
else:
    reference_plate = sorted(reference_plate)
reference_plate = reference_plate[0]

conversion_factors = {}
conversion_factors[reference_plate] = 1.0  # baseline, exp(0)
for term, coef in plate_effects.items():
    plate_number = term.replace('C(Plate)[T.', '').replace(']', '')
    # Compute exp(effect) for multiplicative conversion factor
    conversion_factors[plate_number] = np.exp(coef)

# Display conversion factors for each plate
conv_factors_df = pd.DataFrame(
    list(conversion_factors.items()),
    columns=['Plate', 'conversion_factor']
)
print("Conversion factors for each plate (to equalize them):")
print(conv_factors_df)

# If you want to 'normalize' each log_ratio, you can subtract the plate effect,
# or equivalently divide each RatioLightToHeavy by its conversion factor:
df['RatioLightToHeavy_adj'] = [
    r / conversion_factors[str(int(p)) if str(int(p)) in conversion_factors else str(p)]
    for r, p in zip(df['RatioLightToHeavy'], df['Plate'])
]


In [ ]:
# Map conv_factors_df to skyline_merge by 'Plate' and 'characteristics[Plate]'
# Ensure both columns are comparable types (str or int)

# Convert 'Plate' in conv_factors_df and 'characteristics[Plate]' in skyline_merge to string for joining
conv_factors_df['Plate_str'] = conv_factors_df['Plate'].astype(str)
skyline_merge['characteristics[Plate]_str'] = skyline_merge['characteristics[Plate]'].astype(str)

# Merge conversion factors into skyline_merge
skyline_merge_adj = pd.merge(
    skyline_merge,
    conv_factors_df[['Plate_str', 'conversion_factor']],
    left_on='characteristics[Plate]_str',
    right_on='Plate_str',
    how='left'
)

# Calculate adjusted RatioLightToHeavy
skyline_merge_adj['RatioLightToHeavy_adj'] = skyline_merge_adj['RatioLightToHeavy'] / skyline_merge_adj['conversion_factor']


In [ ]:
# From skyline_merge_adj, filter to pool samples
pool_data_adj = skyline_merge_adj[skyline_merge_adj['characteristics[Sample]'] == 'Pool'].copy()

# If your peptide column is 'Peptide Sequence', rename it once for convenience
if 'Peptide Sequence' in pool_data_adj.columns and 'Peptide' not in pool_data_adj.columns:
    pool_data_adj = pool_data_adj.rename(columns={'Peptide Sequence': 'Peptide'})

# Filter to only peptides in selected_peptides
pool_data_adj = pool_data_adj[pool_data_adj['Peptide'].isin(selected_peptides)].copy() 



In [ ]:

# Plot boxplot of log-transformed RatioLightToHeavy_adj for each Replicate, colored by Plate,
# with hidden x-axis labels and in the style of the previous plot format
df_sorted_adj = pool_data_adj.sort_values('characteristics[Plate]')
plt.figure(figsize=(12, 8))
sns.boxplot(
    x='Replicate',
    y='RatioLightToHeavy_adj',
    data=df_sorted_adj,
    hue='characteristics[Plate]'
)
plt.title('Boxplot of log(RatioLightToHeavy_adj) for Pool samples by Replicate (colored by Plate)')
plt.xlabel('')
plt.ylabel('log(RatioLightToHeavy_adj)')
plt.yscale('log')
plt.xticks([], [])  # Remove x tick labels and marks
plt.legend(title='Plate', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:

# Add new column call qRePS which is the string before _ in Protein Name col
skyline_merge_adj['qRePS'] = skyline_merge_adj['Protein Name'].str.split('_').str[0]

In [ ]:
# Merge qREPs_spike_levels, to skyline_merge_adj by qRePs
skyline_merge_adj = pd.merge(skyline_merge_adj, qREPs_spike_levels, on=['qRePS'], how='left')
# Add new column call qRePs which is the string before _ in Protein Name col

# Add new column called Protein conc [pmol] which is equal to RatioLightToHeavy_adj * Amount Per well [pmol]
skyline_merge_adj['Protein conc [pmol]'] = skyline_merge_adj['RatioLightToHeavy_adj'] * skyline_merge_adj['Amount per well [pmol]']

# Round Protein conc [pmol] to 4 decimal places
skyline_merge_adj['Protein conc [pmol]'] = skyline_merge_adj['Protein conc [pmol]'].round(4)


In [ ]:
# Export wide format for qREPs


# Select columns to export
export_cols = ['Replicate', 'Peptide Sequence', 'Protein Name', 'qRePS', 'Protein conc [pmol]']

# Pivot the dataframe to wide format
skyline_merge_adj_wide = skyline_merge_adj.pivot(
    index=['qRePS', 'Peptide Sequence', 'Protein Name'],
    columns='Replicate',
    values='Protein conc [pmol]'
)

# Export to csv status
skyline_merge_adj_wide.to_csv('export/MARTHA_conc_normalized.csv', index=True)


